In [ ]:
import sys
import json
from pathlib import Path
import base64

from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions,
    TableStructureOptions,
    TableFormerMode,
)

from langchain_docling import DoclingLoader
from langchain_docling.loader import ExportType

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

In [ ]:
import sys
import importlib

sys.path.insert(0, r"D:\Final_GRAG")

import src.text_cleaning
import src.report_utils
importlib.reload(src.text_cleaning)

from src.text_cleaning import (
    clean_text_chunk,
    remove_non_informative_chunks,
    clean_table_dataframe,
    extract_units_from_column,
    is_informative_image,
    standardize_page_numbers,
)

In [ ]:
BASE_DIR = Path(r"D:\Final_GRAG")

REPORT_PATH = BASE_DIR / "reports" / "VCB_Sustainability_report_2024.pdf"
REPORT_ID = "VCB2024"

REPORT_OUTPUT_DIR = BASE_DIR / "metadata" / "report_units" / REPORT_ID
IMAGE_OUTPUT_DIR = REPORT_OUTPUT_DIR / "images"
TABLE_OUTPUT_DIR = REPORT_OUTPUT_DIR / "tables"

REPORT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TABLE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import os
os.environ["TORCHDYNAMO_DISABLE"] = "1"

In [ ]:
pdf_pipeline_options = PdfPipelineOptions(
    # Tables
    do_table_structure=True,
    table_structure_options=TableStructureOptions(
        do_cell_matching=True,
        mode=TableFormerMode.ACCURATE,
    ),
    # Images: extraction + classification + descriptions
    generate_picture_images=True,
    do_picture_classification=True,
    do_picture_description=True, 
)

In [ ]:
converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pdf_pipeline_options)
    }
)

## PDF Conversion

In [ ]:
result = converter.convert(str(REPORT_PATH))
doc = result.document

In [ ]:
# doc_markdown = doc.export_to_markdown()
# print(doc_markdown[:100])

## Text Processing

In [ ]:
loader = DoclingLoader(
    file_path=str(REPORT_PATH),
    converter=converter,
    export_type=ExportType.DOC_CHUNKS,
)

docs = loader.load()

print(len(docs))

In [ ]:
print(json.dumps(docs[0].metadata, indent=2))

In [ ]:
final_text_chunks = []

for i, lc_doc in enumerate(docs):

    # Extract metadata
    meta = lc_doc.metadata
    dl_meta = meta.get("dl_meta", {})
    headings = dl_meta.get("headings", [])

    # Extract page numbers + bboxes
    page_numbers = set()
    bboxes = []
    doc_items = dl_meta.get("doc_items", [])
    for doc_item in doc_items:
        provs = doc_item.get("prov", [])
        for prov in provs:
            page_no = prov.get("page_no")
            if page_no is not None:
                page_numbers.add(page_no)
            bbox = prov.get("bbox")
            if bbox:
                bboxes.append(bbox)

    page_numbers_list = sorted(page_numbers)

    # Chunk dict
    final_text_chunks.append({
        "chunk_id": f"{REPORT_ID}_txt_{i}",
        "content_text": lc_doc.page_content,
        "chunk_type": "text",
        "modality": "text",
        "report_id": REPORT_ID,
        "page_numbers": page_numbers_list,
        "bboxes": bboxes,  
    })

print(len(final_text_chunks))

In [ ]:
# Extract page height
page_heights = {}
if hasattr(doc, "pages"):
    for i, page in doc.pages.items():
        if hasattr(page, "size") and page.size:
            page_heights[i] = page.size.height

cleaned_text_chunks = []
for chunk in final_text_chunks:
    bboxes = chunk.pop("bboxes", [])
    page_numbers = chunk.get("page_numbers", [])
    
    if page_numbers:
        first_page = page_numbers[0]
        page_height = page_heights.get(first_page)
    else:
        page_height = None

    cleaned_text = clean_text_chunk(
        chunk["content_text"],
        bbox=bboxes or None,
        page_height=page_height,
    )
    cleaned_text = remove_non_informative_chunks(cleaned_text)

    if cleaned_text:
        chunk["content_text"] = cleaned_text
        cleaned_text_chunks.append(chunk)

final_text_chunks = cleaned_text_chunks

final_text_chunks[:10]

## Table Processing

In [ ]:
from langchain_ollama import OllamaLLM
llm = OllamaLLM(model="llama3.1:8b", temperature=0.1)

In [ ]:
# Table Summary
TABLE_SUMMARY_PROMPT = ChatPromptTemplate.from_template(
    """You are an expert analyst summarizing tables from sustainability reports of Vietnamese banks.

Given the following table in Markdown format, write a concise summary that captures:

1. **Topic**: What the table is about (e.g., GHG emissions, energy consumption, employee statistics, financial performance, waste management, water usage, community investment, etc.)
2. **Key metrics**: The most important numbers, values, percentages, or trends shown in the table. Include specific figures.
3. **Time period**: Any years or reporting periods covered.
4. **Units**: The measurement units used (e.g., tCO2e, MWh, VND billion, headcount, %).
5. **Notable patterns**: Any significant increases, decreases, comparisons, or highlights visible in the data.

Rules:
- Be factual and precise. Only describe what is present in the table.
- Include specific numbers and values — they are critical for retrieval.
- If the table contains Vietnamese text, preserve key Vietnamese terms alongside their meaning.
- Keep the summary between 3-8 sentences.
- Do NOT use markdown formatting in your summary. Write in plain text.
- Do NOT start with "This table" or "The table". Start directly with the topic or subject.

Table:
{table_markdown}

Summary:"""
)

def generate_table_summary(table_markdown, llm):
    chain = TABLE_SUMMARY_PROMPT | llm | StrOutputParser()
    return chain.invoke({"table_markdown": table_markdown})

In [ ]:
all_table_chunks: list[dict] = []

if hasattr(doc, 'tables') and doc.tables:
    tables = doc.tables
else:
    tables = []

for i, table in enumerate(tables):
    table_id = f"{REPORT_ID}_table_{i}"
    df = table.export_to_dataframe(doc)
    df = clean_table_dataframe(df)
    markdown = df.to_markdown(index=False)

    # Extract units 
    table_units = {}
    for col in df.columns:
        unit = extract_units_from_column(df, col)
        if unit:
            table_units[col] = unit

    # Extract page numbers
    if hasattr(table, 'prov') and table.prov:
        page_numbers = []
        for p in table.prov:
            if hasattr(p, "page_no"):
                page_numbers.append(p.page_no)
    else:
        page_numbers = []

    # Save as CSV 
    table_csv_path = TABLE_OUTPUT_DIR / f"{table_id}.csv"
    df.to_csv(table_csv_path, index=False, encoding='utf-8')
    
    # Save as JSON
    table_json_path = TABLE_OUTPUT_DIR / f"{table_id}.json"
    table_data = {
        "table_id": table_id,
        "num_rows": int(df.shape[0]),
        "num_cols": int(df.shape[1]),
        "columns": list(df.columns),
        "data": df.to_dict(orient='records'),
        "units": table_units,
        "page_numbers": page_numbers,
    }
    with open(table_json_path, 'w', encoding='utf-8') as f:
        json.dump(table_data, f, ensure_ascii=False, indent=2)

    summary_text = generate_table_summary(markdown, llm)
    summary_chunk = {
        "chunk_id": f"{table_id}_summary",
        "content_text": summary_text,
        "chunk_type": "table_summary",
        "modality": "table",
        "report_id": REPORT_ID,
        "page_numbers": page_numbers,
        "column_headers": json.dumps(list(df.columns)),
        "table_units": json.dumps(table_units),
    }
    all_table_chunks.append(summary_chunk)

In [ ]:
all_table_chunks

## Image Processing

In [ ]:
vlm = OllamaLLM(model="llava:7b", temperature=0.1)

In [ ]:
IMAGE_DESCRIPTION_PROMPT = ChatPromptTemplate.from_template(
    """You are an expert analyst describing images from sustainability reports of Vietnamese banks.

The image has been classified as: {image_class}

Describe this image in detail, capturing:

1. **Type**: What kind of visual this is (e.g., bar chart, pie chart, line graph, flowchart, organizational diagram, infographic, photograph, logo, map, etc.)
2. **Subject**: What the image is about (e.g., GHG emissions trend, organizational structure, CSR activities, ESG framework, revenue breakdown, gender diversity, community engagement, etc.)
3. **Data and figures**: If the image contains any numbers, percentages, values, labels, or data points, extract and include them. Be as specific as possible.
4. **Text content**: If there is any text visible in the image (titles, labels, legends, annotations), transcribe it. Preserve Vietnamese terms alongside their meaning if applicable.
5. **Visual patterns**: Describe any notable trends, comparisons, proportions, or highlighted elements visible in the image.

Rules:
- Be factual and precise. Only describe what you can actually see in the image.
- Include specific numbers, labels, and values whenever visible — they are critical for retrieval.
- If text in the image is in Vietnamese, transcribe it and provide a brief English interpretation.
- Keep the description between 3-8 sentences.
- Do NOT use markdown formatting. Write in plain text.
- Do NOT start with "This image" or "The image". Start directly with the type or subject.
- If the image is purely decorative (e.g., background pattern, border, watermark), simply state: "Decorative visual element with no informative content."

Description:"""
)

def generate_image_description(image_path, image_class):
    # Đọc và mã hóa ảnh sang base64
    with open(image_path, "rb") as img_file:
        img_base64 = base64.b64encode(img_file.read()).decode("utf-8")

    # Tạo prompt input
    prompt_vars = {"image_class": image_class or "Unknown"}

    # Định nghĩa hàm gọi VLM với prompt và ảnh
    def call_vlm(prompt):
        prompt_text = prompt.to_string()
        return vlm.invoke(prompt_text, images=[img_base64])

    # Xây dựng chuỗi xử lý
    chain = (
        IMAGE_DESCRIPTION_PROMPT
        | RunnableLambda(call_vlm)
        | StrOutputParser()
    )

    # Gọi chuỗi và trả về kết quả
    return chain.invoke(prompt_vars).strip()

In [ ]:
all_image_chunks: list[dict] = []

if hasattr(doc, "pictures") and doc.pictures:
    pictures = doc.pictures
else:
    pictures = []

for i, picture in enumerate(pictures):
    image_id = f"{REPORT_ID}_img_{i}"    
    img = picture.get_image(doc)
    w, h = img.size

    if not is_informative_image(img, min_width=150, min_height=150, min_area=20000):
        continue

    # Image classification
    classification = ""
    if picture.meta and picture.meta.classification and picture.meta.classification.predictions:
        top_class = picture.meta.classification.predictions[0]
        classification = top_class.class_name

    # Save image
    image_filename = f"{image_id}.png"
    image_path = IMAGE_OUTPUT_DIR / image_filename

    save_img = img.convert("RGB") if img.mode != "RGB" else img
    save_img.save(str(image_path))

    # Prov 
    if picture.prov:
        page_numbers = []
        for p in picture.prov:
            if hasattr(p, "page_no"):
                page_numbers.append(p.page_no)
    else:
        page_numbers = []

    # Generate VLM description
    vlm_description = generate_image_description(str(image_path), classification)

    # Image dict
    all_image_chunks.append({
        "chunk_id": image_id,
        "content_text": vlm_description,
        "chunk_type": "image_description",
        "modality": "image",
        "report_id": REPORT_ID,
        "page_numbers": page_numbers,
        "image_id": image_id,
        "image_path": str(image_path.relative_to(BASE_DIR)),
        "image_classification": classification,
    })

print(len(all_image_chunks))

## Embedding (BGE-M3)

In [ ]:
# Combine text, table, and image chunks thành 1 list 
all_chunks: list[dict] = []

all_chunks.extend(final_text_chunks)
all_chunks.extend(all_table_chunks)
all_chunks.extend(all_image_chunks)

standardize_page_numbers(all_chunks)

# Assign chunk indices
for i, chunk in enumerate(all_chunks):
    chunk["chunk_index"] = i

print(len(all_chunks))

In [ ]:
# Re-index
for i, chunk in enumerate(all_chunks):
    chunk["chunk_index"] = i

print(len(all_chunks))

In [ ]:
from src.embedding_utils import BGEM3Encoder

encoder = BGEM3Encoder()

all_chunks = encoder.encode_chunks(
    all_chunks,
    batch_size=32,
    max_length=8192,
)

In [ ]:
all_chunks[:1]

In [ ]:
output_path = REPORT_OUTPUT_DIR / "report_chunks.json"

# Save all chunks
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(all_chunks, f, ensure_ascii=False, indent=2)

print(f"Saved {len(all_chunks)}")